# 4.0.0-Merge objects by unifying genomic coordinates of peaks

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
options(warn = -1, verbose = FALSE)
suppressMessages(suppressWarnings({
library(Seurat)
library(Signac)
library(tidyverse)
library(GenomicRanges)
library(GenomeInfoDb)
library(ggpubr)
library(EnsDb.Hsapiens.v86)
library(BSgenome.Hsapiens.UCSC.hg38)
library(here)
library(glue)
}))

## Parameters

In [ ]:
#here::dr_here(show_reason = TRUE)
source(here::here("utils/bin.R"))
source(here::here("utils/paths.R"))

"{int}/{plt_dir}" %>%
    glue::glue() %>%
    here::here() %>%
    dir.create(path = .,
               showWarnings = FALSE,
               recursive = TRUE)
"{int}/{robj_dir}" %>%
    glue::glue() %>%
    here::here() %>%
    dir.create(path = .,
               showWarnings = FALSE,
               recursive = TRUE)

set.seed(0)

## Functions

In [ ]:
get_individual_objects <- function(path, info) {
  rds_list <- Sys.glob(file.path(glue::glue("{path}/level_2"), glue::glue("3.3.0-l2_anno_*.rds")))
  list_obj <- lapply(rds_list, function(rds) {
    # read seurat obj
    obj <- readRDS(rds)
    DefaultAssay(obj) <- "ATAC"
      
    # get information from path
    splited <- str_split(rds, "/")[[1]]
    filename <- rev(splited)[1]
    subproj <- rev(splited)[7]
    patient <- gsub("3.3.0-l2_anno_", "", filename)
    patient <- gsub(".rds", "", patient)
    patient <- gsub("-", "_", patient)
      
    # add metadata
    obj$clusters_donor <- Idents(obj)
    obj$barcode <- rownames(obj@meta.data)
    obj$path_obj <- path
    obj$subproj <- subproj
    obj$patient <- patient
    return(obj)
   })
  return(list_obj)
}

## Load data

### Read cohort metadata

In [ ]:
df_info <- read_csv(here(glue("utils/info_samples.csv")))
df_info$donor_id <- gsub("-", "_", df_info$donor_id) 
df_info = df_info[(df_info["modality"] == "multiome" ) & (df_info["quality"] != "Bad"), ]

### Read processed individual objects (from sample-level analysis)

In [ ]:
# first batch AVGUSTINOVA 02 03
source(here("AVGUSTINOVA_02_03/multiome_analysis/misc/paths.R"))
robj1_path <- here(glue("{clust}/{robj_dir}"))

# second batch AVGUSTINOVA 04 05
source(here("AVGUSTINOVA_04_05/multiome_analysis/misc/paths.R"))
robj2_path <- here(glue("{clust}/{robj_dir}"))

# third batch AVGUSTINOVA 10 11
source(here("AVGUSTINOVA_10_11/multiome_analysis/misc/paths.R"))
robj3_path <- here(glue("{clust}/{robj_dir}"))

# fourth batch AVGUSTINOVA 12 13
source(here("AVGUSTINOVA_12_13/multiome_analysis/misc/paths.R"))
robj4_path <- here(glue("{clust}/{robj_dir}"))

# fifth batch AVGUSTINOVA 24 25
source(here("AVGUSTINOVA_24_25/multiome_analysis/misc/paths.R"))
robj5_path <- here(glue("{clust}/{robj_dir}"))

# sixth batch AVGUSTINOVA 22 23
source(here("AVGUSTINOVA_22_23/multiome_analysis/misc/paths.R"))
robj6_path <- here(glue("{clust}/{robj_dir}"))

# seventh batch AVGUSTINOVA 29 30
source(here("AVGUSTINOVA_29_30/multiome_analysis/misc/paths.R"))
robj7_path <- here(glue("{clust}/{robj_dir}"))

In [ ]:
source(here::here("utils/paths.R"))
"{int}/{robj_dir}" %>%
  glue::glue() %>%
  here::here() %>%
  dir.create(path = .,
           showWarnings = FALSE,
           recursive = TRUE)
set.seed(0)

In [ ]:
l1 <- get_individual_objects(robj1_path, df_info)
l2 <- get_individual_objects(robj2_path, df_info)
l3 <- get_individual_objects(robj3_path, df_info)
l4 <- get_individual_objects(robj4_path, df_info)
l5 <- get_individual_objects(robj5_path, df_info)
l6 <- get_individual_objects(robj6_path, df_info)
l7 <- get_individual_objects(robj7_path, df_info)

In [ ]:
list_objects <- unlist(list(l1,l2,l3,l4,l5,l6,l7))

### Get annotations for Hg38 genome build

This is executed locally, saved and then uploaded again for its use

In [ ]:
# library(EnsDb.Hsapiens.v86)
# library(BSgenome.Hsapiens.UCSC.hg38)
# # get gene annotations for hg38
# annotation <- GetGRangesFromEnsDb(ensdb = EnsDb.Hsapiens.v86)
# seqlevelsStyle(annotation) <- "UCSC"
# genome(annotation) <- "hg38"
# saveRDS(annotation, here::here(glue::glue("external_data/annotation.rds")))

In [ ]:
annotation <- readRDS(here::here(glue::glue("external_data/annotation.rds"))) 

## Unify peaks

In [ ]:
if (TRUE) {
combined_peaks <- UnifyPeaks(object.list = list_objects, mode = "reduce")
combined_peaks
table(combined_peaks@seqnames)

# Keep only standard chromosomes
combined_peaks <- keepStandardChromosomes(
  combined_peaks,
  pruning.mode = "coarse"
)
table(combined_peaks@seqnames)

# Remove peaks with outlier widths
peak_widths <- width(combined_peaks)
ggviolin(peak_widths, add = "boxplot", fill = "gray") +
  scale_y_log10() +
  geom_hline(yintercept = c(20, 10000), linetype = "dashed", col = "black")
combined_peaks <- combined_peaks[peak_widths < 10000 & peak_widths > 20]
combined_peaks
}

## Quantify new peaks 

In [ ]:
patient_names <- lapply(list_objects, function(obj) {
    unique(obj$patient)
    })

In [ ]:
patient_names

In [ ]:
if (TRUE) {
list_objects <- lapply(list_objects, function(obj) {
  
  # get path fragments and redo path for cluster file system
  obj <- change_frag_path(obj, "ATAC")
  
  # quantify counts in each peak
  counts <- FeatureMatrix(
    fragments = Fragments(obj),
    features = combined_peaks,
    cells = colnames(obj))

  # create a new assay using the unified peak set and add it to the Seurat object
  obj[["ATAC_unified"]] <- CreateChromatinAssay(
    counts = counts,
    fragments = Fragments(obj),
    annotation = annotation) 
  return(obj)
})
}

In [ ]:
list_objects <- setNames(list_objects, patient_names)

In [ ]:
saveRDS(list_objects, here(glue("{int}/{robj_dir}/4.0.0-list_objects_unify.rds")))

In [ ]:
list_objects <-  readRDS(here(glue("{int}/{robj_dir}/4.0.0-list_objects_unify.rds")))

In [ ]:
list_objects

In [ ]:
combined_obj <- merge(
  x = list_objects[[1]],
  y = list_objects[2:length(list_objects)],
  add.cell.ids = c(names(list_objects))
)

In [ ]:
saveRDS(combined_obj, here(glue("{int}/{robj_dir}/4.0.0-combined_object_unify.rds")))

## Remove lowly expressed genes 

We apply this here to lighter the object

In [ ]:
if (TRUE) {
keep_genes <- data.frame(rowSums(as.matrix(combined_obj[['RNA']]@counts)) >= 5)
colnames(keep_genes) <- "logical"
keep_genes <- subset(keep_genes, logical==TRUE)
combined_obj[['RNA']]@counts <- combined_obj[['RNA']]@counts[rownames(keep_genes), ]
combined_obj[['RNA']]@data <- combined_obj[['RNA']]@data[rownames(keep_genes), ]

# remove individual peak calls
DefaultAssay(combined_obj) <- "ATAC_unified"
combined_obj[["ATAC"]] <- NULL
}

In [ ]:
saveRDS(combined_obj, here(glue("{int}/{robj_dir}/4.0.0-combined_object_unify.rds")))

combined_obj <- readRDS(here(glue("{int}/{robj_dir}/4.0.0-combined_object_unify.rds")))

In [ ]:
combined_obj

In [ ]:
Fragments(list_objects$p5)

In [ ]:
list_objects <- lapply(list_objects, function(obj) {
    
  print(unique(obj@meta.data$patient))
  
  # filter by used genes
  DefaultAssay(obj) <- "RNA"
  filter_genes <- intersect(rownames(obj[['RNA']]@counts),rownames(keep_genes))
  obj[['RNA']]@counts <- obj[['RNA']]@counts[filter_genes, ]
  obj[['RNA']]@data <- obj[['RNA']]@data[filter_genes, ]
  
  return(obj)
})

In [ ]:
list_objects

In [ ]:
list_objects <- setNames(list_objects, patient_names)

In [ ]:
list_objects

In [ ]:
# free some space
remove(keep_genes)
saveRDS(list_objects, here(glue("{int}/{robj_dir}/4.0.0-list_objects_unify.rds")))
remove(list_objects)

## Merge fragment files

Follow [Signac instructions](https://stuartlab.org/signac/0.2/articles/merging.html) to merge fragment files adding the patient to the beginning of the barcode. Merged fragment file should be then "loaded" into the combined object.

In [ ]:
frag_path <- here::here(glue::glue("{int}/frag_merged/fragments.tsv.gz"))
fragments <- CreateFragmentObject(path = frag_path, cells = colnames(combined_obj), validate.fragments = TRUE)
Fragments(combined_obj@assays$ATAC_unified) <- NULL
Fragments(combined_obj@assays$ATAC_unified) <- fragments
saveRDS(combined_obj, here::here(glue::glue("{int}/{robj_dir}/4.0.0-combined_object_unify.rds")))

## Session Info

In [ ]:
sessionInfo()